# Chapter 56 — Document Loaders: PDF, TXT, DOCX and CSV

Project
Build a loader that reads existing input files in four different formats and converts their content into one consistent record structure. You will use the sample files supplied with this chapter; this notebook does not create the input documents.
The sample pack contains a fictional cross-border transaction review in TXT, PDF and DOCX formats, plus a transaction CSV. The ZIP also contains a JSONL example of normalized content. We will generate our own normalized JSONL from the files we actually load, and keep it separate from the supplied example.

In [2]:
%pip install -q pypdf python-docx pandas

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# 2. Locate the downloaded files

In [56]:
from pathlib import Path
from zipfile import ZipFile
import json
import hashlib
import pandas as pd
from pypdf import PdfReader
from docx import Document
from IPython.display import display

In [57]:
DATA_DIR = Path("chapter_56_input")
OUTPUT_DIR = Path("chapter_56_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".txt", ".pdf", ".docx", ".csv"}

In [58]:
print("Notebook working folder:", Path.cwd())
print("Input folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


Notebook working folder: /Users/vimal/Documents/GitHub/Agentic_AI
Input folder: /Users/vimal/Documents/GitHub/Agentic_AI/chapter_56_input
Output folder: /Users/vimal/Documents/GitHub/Agentic_AI/chapter_56_output


In [59]:
input_files = sorted(path for path in DATA_DIR.iterdir()
if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS)

In [60]:
found_extensions = {path.suffix.lower() for path in input_files}


In [61]:
required_extensions = {".txt", ".pdf", ".docx", ".csv"}


In [62]:
missing_extensions = sorted(required_extensions - found_extensions)


In [63]:
if missing_extensions:
    missing_types = ", ".join(missing_extensions)
    raise FileNotFoundError(
        "The input folder is missing one or more required file types: "
        f"{missing_types}. Put your individually downloaded TXT, PDF, DOCX and CSV "
        "files into the folder shown above, then rerun this cell. "
        "The filenames may contain spaces and do not have to match a specific name."
    )


In [64]:
print(f"Supported source files discovered: {len(input_files)}")
for path in input_files:
    print(" -", path.name)


Supported source files discovered: 4
 - Chapter 56 - Document Loaders - Review.docx
 - Chapter 56 - Document Loaders - Review.pdf
 - Chapter 56 - Document Loaders - Review.txt
 - Chapter 56 - Document Loaders - Transactions.csv


# 3. Inspect the input files before parsing

In [65]:
input_files = sorted(
    path for path in DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

In [66]:
input_files

[PosixPath('chapter_56_input/Chapter 56 - Document Loaders - Review.docx'),
 PosixPath('chapter_56_input/Chapter 56 - Document Loaders - Review.pdf'),
 PosixPath('chapter_56_input/Chapter 56 - Document Loaders - Review.txt'),
 PosixPath('chapter_56_input/Chapter 56 - Document Loaders - Transactions.csv')]

In [67]:

inventory = []
for path in input_files:
    raw_bytes = path.read_bytes()
    inventory.append({
        "file_name": path.name,
        "file_type": path.suffix.lower().lstrip("."),
        "size_bytes": len(raw_bytes),
        "sha256_prefix": hashlib.sha256(raw_bytes).hexdigest()[:16],
    })

In [68]:
inventory

[{'file_name': 'Chapter 56 - Document Loaders - Review.docx',
  'file_type': 'docx',
  'size_bytes': 37369,
  'sha256_prefix': '165889a3538dd371'},
 {'file_name': 'Chapter 56 - Document Loaders - Review.pdf',
  'file_type': 'pdf',
  'size_bytes': 27676,
  'sha256_prefix': 'ae594be0d1ace0ba'},
 {'file_name': 'Chapter 56 - Document Loaders - Review.txt',
  'file_type': 'txt',
  'size_bytes': 1072,
  'sha256_prefix': 'c375b891518dccb8'},
 {'file_name': 'Chapter 56 - Document Loaders - Transactions.csv',
  'file_type': 'csv',
  'size_bytes': 267,
  'sha256_prefix': 'fa005adbc668ae8a'}]

In [69]:
display(pd.DataFrame(inventory))


,file_name,file_type,size_bytes,sha256_prefix
0,Chapter 56 - Document Loaders - Review.docx,docx,37369,165889a3538dd371
1,Chapter 56 - Document Loaders - Review.pdf,pdf,27676,ae594be0d1ace0ba
2,Chapter 56 - Document Loaders - Review.txt,txt,1072,c375b891518dccb8
3,Chapter 56 - Document Loaders - Transactions.csv,csv,267,fa005adbc668ae8a


In [70]:
print(f"Supported input files discovered: {len(input_files)}")
if not input_files:
    raise RuntimeError("No TXT, PDF, DOCX or CSV files were found in DATA_DIR.")

Supported input files discovered: 4


# 4. Define one common record schema

In [71]:
def make_record(source_file, file_type, location, text, metadata=None):
    """Create a record with the shared schema used by all loaders."""
    return {
        "source_file": str(source_file),
        "file_type": str(file_type).lower(),
        "location": location,
        "text": str(text).strip(),
        "metadata": metadata if metadata is not None else {},
    }

make_record(
    "example.txt", "txt", {"line": 1}, "Example text",
    {"encoding": "utf-8"}
)

{'source_file': 'example.txt',
 'file_type': 'txt',
 'location': {'line': 1},
 'text': 'Example text',
 'metadata': {'encoding': 'utf-8'}}

# 5. TXT loader — one record per non-empty line
Text files are simple to read, but encoding problems can still occur. Here we use UTF-8 and replace characters that cannot be decoded rather than crashing on one unusual character.

In [72]:
def load_txt(path):
    path = Path(path)
    records = []

    with path.open("r", encoding="utf-8-sig", errors="replace") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip():
                records.append(make_record(
                    source_file=path.name,
                    file_type="txt",
                    location={"line": line_number},
                    text=line,
                    metadata={
                        "encoding": "utf-8-sig",
                        "extraction_method": "line-by-line",
                    },
                ))

    return records

In [73]:
txt_path = next(path for path in input_files if path.suffix.lower() == ".txt")
txt_preview = load_txt(txt_path)
print(f"TXT loader returned {len(txt_preview)} record(s) from {txt_path.name}")
display(pd.DataFrame(txt_preview).head(8))

TXT loader returned 16 record(s) from Chapter 56 - Document Loaders - Review.txt


,source_file,file_type,location,text,metadata
0,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 1},Cross-Border Transaction Review,"{'encoding': 'utf-8-sig', 'extraction_method':..."
1,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 3},Purpose: This fictional example demonstrates h...,"{'encoding': 'utf-8-sig', 'extraction_method':..."
2,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 4},Case ID: CBR-2026-0142,"{'encoding': 'utf-8-sig', 'extraction_method':..."
3,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 5},Customer type: Corporate trading company,"{'encoding': 'utf-8-sig', 'extraction_method':..."
4,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 6},Review period: 12–14 October 2026,"{'encoding': 'utf-8-sig', 'extraction_method':..."
5,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 7},Transactions reviewed: 3,"{'encoding': 'utf-8-sig', 'extraction_method':..."
6,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 8},"Total value: USD 525,000","{'encoding': 'utf-8-sig', 'extraction_method':..."
7,Chapter 56 - Document Loaders - Review.txt,txt,{'line': 9},Originator countries: United States and United...,"{'encoding': 'utf-8-sig', 'extraction_method':..."


# 6. PDF loader — one record per page

In [75]:
def load_pdf(path):
    path = Path(path)
    reader = PdfReader(str(path))
    records = []
    page_count = len(reader.pages)

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = (page.extract_text() or "").strip()
        
        metadata = { 
            "pdf_page_count": page_count,
            "extraction_method": "pypdf text extraction"
        }
        
        if not page_text:
            metadata["warning"] = (
                "No selectable text extracted, the page may be scanned. "
                "Inspect the page or consider OCR."
            )

        # Fixed the function call syntax and the page location logic
        records.append(make_record(
            source_file=path.name, 
            file_type="pdf",
            location={"page": page_number}, 
            text=page_text,
            metadata=metadata
        ))
        
    return records

In [76]:
pdf_path = next(path for path in input_files if path.suffix.lower() == ".pdf")
pdf_preview = load_pdf(pdf_path)
print(f"PDF loader returned {len(pdf_preview)} page record(s) from {pdf_path.name}")
display(pd.DataFrame(pdf_preview)[["source_file", "location", "text", "metadata"]])

PDF loader returned 1 page record(s) from Chapter 56 - Document Loaders - Review.pdf


,source_file,location,text,metadata
0,Chapter 56 - Document Loaders - Review.pdf,{'page': 1},Cross-Border Transaction Review\nPurpose: This...,"{'pdf_page_count': 1, 'extraction_method': 'py..."


# 7. DOCX loader — paragraphs and table rows

In [89]:
def load_docs(path):
    path = Path(path)
    document = Document(str(path))
    record = []

    for paragraph_number, paragraph in enumerate(document.paragraphs, start = 1):
        if paragraph.text.strip():
            record.append(make_record(
                source_file=path.name,
                file_type = "docx",
                location={"paragraph": paragraph_number},
                text = paragraph.text,
                metadata={"element_type"}
            ))
    for table_number, table in enumerate(document.tables, start = 1):
        for row_number, row in enumerate(table.rows, start = 1):
            cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            row_text = " | ".join(cells)
            if row_text.strip():
                record.append(make_record(
                    source_file=path.name, 
                    file_type="docx",
                    location={"table" : table_number, "row" : row_number},
                    text = row_text,
                    metadata = {"element_type" : "table_row", "cell_count" : len(cells,)},))
    return record
                

In [ ]:
docx_path = next(path for path in input_files if path.suffix.lower() == ".docx")
docx_records = load_docs(docx_path)
print(f"Loaded {len(docx_records)} records from {docx_path.name}")
display(pd.DataFrame(docx_records))

Loaded 16 records from Chapter 56 - Document Loaders - Review.docx


,source_file,file_type,location,text,metadata
0,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 1},Cross-Border Transaction Review,{element_type}
1,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 2},Purpose: This fictional example demonstrates h...,{element_type}
2,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 3},Case ID: CBR-2026-0142,{element_type}
3,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 4},Customer type: Corporate trading company,{element_type}
4,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 5},Review period: 12–14 October 2026,{element_type}
5,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 6},Transactions reviewed: 3,{element_type}
6,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 7},"Total value: USD 525,000",{element_type}
7,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 8},Originator countries: United States and United...,{element_type}
8,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 9},Beneficiary countries: United Arab Emirates an...,{element_type}
9,Chapter 56 - Document Loaders - Review.docx,docx,{'paragraph': 10},Review note: The transfers require assessment ...,{element_type}


# 8. CSV loader — one record per data row

In [94]:
def load_csv(path):
    path = Path(path)
        # Parse the CSV into columns and rows; do not send it through load_txt().
    dataframe = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    records = []
    columns = [str(column) for column in dataframe.columns]

    for dataframe_index, row in dataframe.iterrows():
        physical_row_number = int(dataframe_index) + 2

        row_text = "; ".join(f"{columns} : {row[columns]}" for column in dataframe.columns)

        records.append(make_record(
            source_file=path.name,
            file_type="csv",  # Deliberately hard-coded to the CSV format.
            location={"row": physical_row_number},
            text=row_text,
            metadata={
                "columns": columns,
                "extraction_method": "pandas read_csv; one record per data row",
            },
        ))

    return records






In [95]:
csv_path = next(path for path in input_files if path.suffix.lower() == ".csv")
csv_preview = load_csv(csv_path)
print(f"CSV loader returned {len(csv_preview)} data-row record(s) from {csv_path.name}")
display(pd.DataFrame(csv_preview))

CSV loader returned 3 data-row record(s) from Chapter 56 - Document Loaders - Transactions.csv


,source_file,file_type,location,text,metadata
0,Chapter 56 - Document Loaders - Transactions.csv,csv,{'row': 2},"['Transaction ID', 'Date', 'Originator country...","{'columns': ['Transaction ID', 'Date', 'Origin..."
1,Chapter 56 - Document Loaders - Transactions.csv,csv,{'row': 3},"['Transaction ID', 'Date', 'Originator country...","{'columns': ['Transaction ID', 'Date', 'Origin..."
2,Chapter 56 - Document Loaders - Transactions.csv,csv,{'row': 4},"['Transaction ID', 'Date', 'Originator country...","{'columns': ['Transaction ID', 'Date', 'Origin..."


# 9. Unified loader — dispatch by extension

This is the central dispatch table. load_document() inspects the file extension and selects exactly one loader. It also checks each record's file_type after parsing. If a CSV file is accidentally sent through the TXT loader, this check raises an error immediately instead of quietly returning misleading data.

In [98]:
LOADERS = {
    ".txt": load_txt,
    ".pdf": load_pdf,
    ".docx": load_docs,
    ".csv": load_csv,
}

In [104]:
EXPECTED_TYPE_BY_EXTENSION = {
    ".txt": "txt",
    ".pdf": "pdf",
    ".docx": "docx",
    ".csv": "csv",
}

In [107]:
def load_document(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError("Input file does not exist")

    extension = path.suffix.lower()
    loader = LOADERS.get(extension)

    if loader is None:
        supported = ", ".join(sorted(LOADERS))
        raise ValueError("unsupported extension")

    expected_type = EXPECTED_TYPE_BY_EXTENSION[extension]

    records = loader(path)

    mismatched = [
        record for record in records
        if record.get("file_type") != expected_type
    ]
    if mismatched:
        raise ValueError(
            f"Loader mismatch for {path.name}: extension {extension!r} requires "
            f"file_type={expected_type!r}, but a record had a different type."
        )

    return records

# Dispatch smoke test: prove that the four extensions call the expected loaders.
for path in input_files:
    records = load_document(path)
    expected_type = EXPECTED_TYPE_BY_EXTENSION[path.suffix.lower()]
    assert all(record["file_type"] == expected_type for record in records)

    


# 10. Process all files and isolate individual failures

In [108]:
def load_folder(paths):
    all_records = []
    report_rows = []

    for path in sorted((Path(p) for p in paths), key=lambda p: p.name.lower()):
        try:
            records = load_document(path)
            all_records.extend(records)
            report_rows.append({
                "source_file": path.name,
                "status": "success",
                "records": len(records),
                "error": "",
            })
        except Exception as exc:
            report_rows.append({
                "source_file": path.name,
                "status": "failed",
                "records": 0,
                "error": f"{type(exc).__name__}: {exc}",
            })

    return all_records, pd.DataFrame(
        report_rows, columns=["source_file", "status", "records", "error"]
    )

all_records, file_report = load_folder(input_files)
display(file_report)
print("Total normalized records:", len(all_records))

if file_report.empty or (file_report["status"] != "success").any():
    print("Review the report above for files that did not load successfully.")


,source_file,status,records,error
0,Chapter 56 - Document Loaders - Review.docx,success,16,
1,Chapter 56 - Document Loaders - Review.pdf,success,1,
2,Chapter 56 - Document Loaders - Review.txt,success,16,
3,Chapter 56 - Document Loaders - Transactions.csv,success,3,


Total normalized records: 36
